# Heavy Resized — поэтапное продолжение HPO

Существующий поиск: `heavy_resized_from_best_60ep_v3`. Готовые trials переиспользуются, незавершённые продолжаются из `last.pt`.

Сначала останови процессы старого поиска, установи новый `automated_hpo.py` и перезапусти kernel. Не изменяй общий пакет, пока из него работает другой HPO.

Запускай ячейки этапов по отдельности. По снимку архива этапы 1 и 2 завершены: после подготовки можно сразу выполнить **этап 3**. Более новые результаты на диске также будут учтены.


## 1. Пути, протокол и CHECKPOINTS

In [4]:
from pathlib import Path
import json

from satchange.automated_hpo import run_automated_parameter_search
from satchange.final_benchmark import run_final_benchmark

cwd = Path.cwd().resolve()
project_root = next(
    path for path in (cwd, *cwd.parents)
    if (path / "src/satchange").is_dir()
)
HPO_ROOT = project_root / "artifacts/hyperparameter_search/heavy/heavy_resized"
SEARCH_NAME = "heavy_resized_from_best_60ep_v3"
SEARCH_DIRECTORY = HPO_ROOT / SEARCH_NAME

PROJECT_ROOT = Path(
    "."
)

# Use the experiment's actual settings, not a YAML edited after it started.
protocol = json.loads(
    (SEARCH_DIRECTORY / "automated_hpo_protocol.json").read_text(encoding="utf-8")
)
HPO_STAGES = protocol["stages"]

# External initial checkpoints must remain the same for this experiment.
# Current incomplete trials use their own last.pt automatically.
CHECKPOINTS = [Path(item["path"]) for item in protocol.get("resume_from", [])]
GPU_IDS = [0, 1, 2, 3]  # Only GPUs allocated to this search.

print("Search:", SEARCH_DIRECTORY)
print("Stages:", [name for name, _ in HPO_STAGES])
print("External CHECKPOINTS:", len(CHECKPOINTS))
for path in CHECKPOINTS:
    print(path)


Search: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3
Stages: ['01_batch_lr', '02_lr', '03_loss', '04_wd_scheduler']
External CHECKPOINTS: 4
artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_000/best.pt
artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_001/best.pt
artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_002/best.pt
artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_003/best.pt


## 2. Общие аргументы — выполнить один раз после каждого Restart Kernel

In [2]:
policy = protocol["early_stopping"] or {}

HPO_OPTIONS = dict(
    stages=HPO_STAGES,
    base_parameters=protocol["base"],
    model_name=protocol["model"],
    dataset_root=Path(protocol["dataset_root"]),
    thresholds=protocol["thresholds"],
    number_of_epochs=protocol["epochs"],
    patience=policy.get("patience"),
    min_epochs=policy.get("min_epochs", 1),
    min_delta=policy.get("min_delta", 0.0),
    output_root=HPO_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=GPU_IDS,
    checkpoint_interval=10,
    resume_incomplete=True,
    resume_from=CHECKPOINTS,
    f1_gap=protocol["gaps"][0],
    iou_gap=protocol["gaps"][1],
    seeds=protocol["seeds"] if protocol["evaluate_seeds"] else None,
    deployment_seed=protocol["deployment_seed"],
)

print("Maximum epochs:", HPO_OPTIONS["number_of_epochs"])
print("Image size:", HPO_OPTIONS["base_parameters"]["image_size"])
print("num_workers:", HPO_OPTIONS["base_parameters"]["num_workers"])
print("Early stopping:", policy)
print("Final seeds:", HPO_OPTIONS["seeds"])


Maximum epochs: 60
Image size: 256
num_workers: 0
Early stopping: {'min_delta': 0.001, 'min_epochs': 40, 'monitor': 'validation_f1', 'patience': 20}
Final seeds: None


## Этап 1. Batch × learning rate

В архиве уже завершён. Эта ячейка прочитает результаты и пересчитает отбор без нового обучения.

In [3]:
stage1 = run_automated_parameter_search(**HPO_OPTIONS, stage="01_batch_lr")
print("Прошли дальше:", len(stage1["survivors"]))
print("Отбор:", stage1["selection_report"])

src/satchange/optimization.py:270: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_000/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:270: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_001/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(
src/satchange/optimization.py:270: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_002/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(


01_batch_lr: new=0, reused=12
01_batch_lr: keep 5/12; best F1=0.737402, IoU=0.584035
Прошли дальше: 5
Отбор: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3/01_batch_lr_selection.json


src/satchange/optimization.py:270: RuntimeWarning: Legacy checkpoint artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_stages_only_es_v1/01_batch_lr_edbd53c55e07/siam_mit_b3_diff/trial_003/best.pt has no dataset path: verify it uses the same dataset.
  warnings.warn(


## Этап 2. Уточнение learning rate

В архиве уже завершён. Эта ячейка прочитает результаты и пересчитает отбор без нового обучения.

In [4]:
stage2 = run_automated_parameter_search(**HPO_OPTIONS, stage="02_lr")
print("Прошли дальше:", len(stage2["survivors"]))
print("Отбор:", stage2["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 5/12; best F1=0.737402, IoU=0.584035
02_lr: new=0, reused=12
02_lr: keep 9/12; best F1=0.745305, IoU=0.594013
Прошли дальше: 9
Отбор: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3/02_lr_selection.json


## Этап 3. Positive weight × BCE/Dice

Начни продолжение с этой ячейки после подготовки. Ранее законченные loss-trials не повторяются; для остальных используется одна очередь на выделенных GPU.

In [5]:
stage3 = run_automated_parameter_search(**HPO_OPTIONS, stage="03_loss")
print("Прошли дальше:", len(stage3["survivors"]))
print("Отбор:", stage3["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 5/12; best F1=0.737402, IoU=0.584035
02_lr: new=0, reused=12
02_lr: keep 9/12; best F1=0.745305, IoU=0.594013
03_loss: new=0, reused=36
03_loss: keep 18/36; best F1=0.775586, IoU=0.633435
Прошли дальше: 18
Отбор: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3/03_loss_selection.json


## Этап 4. Weight decay × scheduler

Запусти после окончания этапа 3. Если он не завершён, функция сообщит об этом и не начнёт предыдущие этапы сама.

In [6]:
stage4 = run_automated_parameter_search(**HPO_OPTIONS, stage="04_wd_scheduler")
print("Прошли дальше:", len(stage4["survivors"]))
print("Отбор:", stage4["selection_report"])

01_batch_lr: new=0, reused=12
01_batch_lr: keep 5/12; best F1=0.737402, IoU=0.584035
02_lr: new=0, reused=12
02_lr: keep 9/12; best F1=0.745305, IoU=0.594013
03_loss: new=0, reused=36
03_loss: keep 18/36; best F1=0.775586, IoU=0.633435
04_wd_scheduler: new=46, reused=26
04_wd_scheduler: queue=46, parallel_limit=4, gpu_ids=[0, 1, 2, 3]
04_wd_scheduler: resume trial 000 after epoch 23
04_wd_scheduler: resume trial 001 after epoch 22
04_wd_scheduler: resume trial 002 after epoch 16
04_wd_scheduler: resume trial 003 after epoch 16
Trials: 46
GPUs: [0, 1, 2, 3]
Single-GPU memory limit: 80000 MB

[START] Trial 000 -> GPU 0 | attempt=1
seed=42 | lr=0.0003 | wd=0.01 | pw=5 | BCE/Dice=0.5/0.5 | batch=4 | image_size=256 | attention_dropout=0.0 | decoder_channels=128 | drop_path_rate=0.1 | dropout=0.0 | encoder_depths=[3, 4, 18, 3] | in_channels=3 | norm_groups=8 | out_channels=1 | pretrained_backbone=False | refinement_channels=[64, 32] | use_checkpointing=False | scheduler=reduce_on_plateau
epo

## Итоговая проверка и отчёт

В протоколе `seeds=None`: эта ячейка только собирает итог по HPO-seed 42. Дополнительные seeds 123/777 не запускаются. Она требует, чтобы все четыре этапа уже были завершены.

In [7]:
summary = run_automated_parameter_search(**HPO_OPTIONS, stage="seeds")
best = summary["selected_by_mean_validation_f1"]
print("Validation F1:", best["mean_f1"], "+/-", best["std_f1"])
print("Validation IoU:", best["mean_iou"], "+/-", best["std_iou"])
print("Parameters:", best["parameters"])
print("Report:", SEARCH_DIRECTORY / "final_validation_summary.json")

01_batch_lr: new=0, reused=12
01_batch_lr: keep 5/12; best F1=0.737402, IoU=0.584035
02_lr: new=0, reused=12
02_lr: keep 9/12; best F1=0.745305, IoU=0.594013
03_loss: new=0, reused=36
03_loss: keep 18/36; best F1=0.775586, IoU=0.633435
04_wd_scheduler: new=0, reused=72
04_wd_scheduler: keep 63/72; best F1=0.775586, IoU=0.633435
Final validation summary: artifacts/hyperparameter_search/heavy/heavy_resized/heavy_resized_from_best_60ep_v3/final_validation_summary.json
Validation F1: 0.7755862537285373 +/- None
Validation IoU: 0.6334347814130006 +/- None
Parameters: {'batch_size': 4, 'bce_weight': 0.5, 'dice_weight': 0.5, 'image_size': 256, 'learning_rate': 0.0006, 'model_parameters': {'attention_dropout': 0.0, 'decoder_channels': 128, 'drop_path_rate': 0.1, 'dropout': 0.0, 'encoder_depths': [3, 4, 18, 3], 'in_channels': 3, 'norm_groups': 8, 'out_channels': 1, 'pretrained_backbone': False, 'refinement_channels': [64, 32], 'use_checkpointing': False}, 'num_workers': 0, 'positive_weight': 10

In [6]:
summary_path = checkpoint.parent / "summary.json"

if summary_path.is_file():
    trial_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    seed = trial_summary["parameters"]["seed"]
else:
    # Резервный вариант для собственного checkpoint без summary.json.
    saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
    seed = saved["config"]["seed"]
    del saved


checkpoint = PROJECT_ROOT / (
    "artifacts/hyperparameter_search/heavy/heavy_resized/"
    "heavy_resized_from_best_60ep_v3/03_loss_4e5a461f5cf3/"
    "siam_mit_b3_diff/trial_007/best.pt"
)


summary = run_final_benchmark(
    variant="heavy_resized",
    checkpoints={int(seed): checkpoint},

    project_root=PROJECT_ROOT,
    dataset_root=PROJECT_ROOT / "data/LEVIR-CD",

    device="auto",
    evaluation_batch_size=1,
    num_workers=0,
    precision="fp32",

    warmup=20,
    repetitions=100,

    strict_validation=True,
    overwrite=False,
)

print(summary["output_dir"])

Working directory: artifacts/final_models/.heavy_resized.benchmark_work_gkmxaycd

heavy_resized | siam_mit_b3_diff | 256x256 | FP32 | cuda:0 | seeds=[42]
validation seed=   42 | threshold=0.85 | F1=0.775655 | IoU=0.633526
Deployment selected BEFORE test: seed=42, threshold=0.85 (highest saved validation F1; not test F1; ties: smallest seed)
test       seed=   42 | threshold=0.85 | F1=0.746320 | IoU=0.595303

Saved: artifacts/final_models/heavy_resized
validation F1 = 0.775655 ± n/a (one seed)
test       F1 = 0.746320 ± n/a (one seed)
Deployment latency: 41.066 ms/pair | 24.351 pairs/s
artifacts/final_models/heavy_resized


In [3]:
from pathlib import Path
import json

import pandas as pd


ROOT = Path(
    "~/projects/"
    "satellite-change-intelligence/artifacts/final_models"
)

rows = []

for model_dir in sorted(ROOT.iterdir()):
    if not model_dir.is_dir() or model_dir.name.startswith("."):
        continue

    validation_file = model_dir / "validation_results.json"
    test_file = model_dir / "final_test_results.json"

    if not validation_file.exists() or not test_file.exists():
        print(f"[SKIP] {model_dir.name}: нет validation/test results")
        continue

    validation = json.loads(validation_file.read_text(encoding="utf-8"))
    test = json.loads(test_file.read_text(encoding="utf-8"))

    rows.append({
        "model": model_dir.name,

        "validation_F1": validation.get("validation_f1_mean"),
        "validation_F1_std": validation.get("validation_f1_std"),
        "validation_IoU": validation.get("validation_iou_mean"),
        "validation_IoU_std": validation.get("validation_iou_std"),

        "test_F1": test.get("test_f1_mean"),
        "test_F1_std": test.get("test_f1_std"),
        "test_IoU": test.get("test_iou_mean"),
        "test_IoU_std": test.get("test_iou_std"),

        "seeds": test.get("seeds"),
    })


df = pd.DataFrame(rows)

# Для удобства сравнения
df = df.sort_values("test_F1", ascending=False).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n=== FINAL MODEL COMPARISON ===\n")

display(
    df.style.format({
        "validation_F1": "{:.6f}",
        "validation_F1_std": lambda x: "—" if pd.isna(x) else f"{x:.6f}",
        "validation_IoU": "{:.6f}",
        "validation_IoU_std": lambda x: "—" if pd.isna(x) else f"{x:.6f}",
        "test_F1": "{:.6f}",
        "test_F1_std": lambda x: "—" if pd.isna(x) else f"{x:.6f}",
        "test_IoU": "{:.6f}",
        "test_IoU_std": lambda x: "—" if pd.isna(x) else f"{x:.6f}",
    })
)


print("\n=== TEST RESULTS PER SEED ===\n")

for model_dir in sorted(ROOT.iterdir()):
    test_file = model_dir / "final_test_results.json"

    if not test_file.exists():
        continue

    data = json.loads(test_file.read_text(encoding="utf-8"))

    print(f"\n{model_dir.name}")

    for result in data.get("results", []):
        print(
            f"  seed={result['seed']:<5} "
            f"F1={result['test_f1']:.6f} | "
            f"IoU={result['test_iou']:.6f} | "
            f"threshold={result.get('validation_threshold')}"
        )


=== FINAL MODEL COMPARISON ===



,model,validation_F1,validation_F1_std,validation_IoU,validation_IoU_std,test_F1,test_F1_std,test_IoU,test_IoU_std,seeds
0,medium_full,0.856658,0.006813,0.749299,0.010388,0.845726,0.007093,0.732734,0.010619,"[42, 123, 777]"
1,light_full,0.814642,0.041413,0.688614,0.058397,0.805383,0.038399,0.675324,0.053558,"[123, 777, 2024]"
2,heavy_resized,0.775655,—,0.633526,—,0.746320,—,0.595303,—,[42]
3,light_resized,0.740551,0.014097,0.588129,0.017786,0.737398,0.017719,0.584240,0.022331,"[42, 123, 777]"
4,medium_resized,0.711009,0.007211,0.551634,0.008696,0.684227,0.022479,0.520314,0.025966,"[42, 123, 777]"



=== TEST RESULTS PER SEED ===


heavy_resized
  seed=42    F1=0.746320 | IoU=0.595303 | threshold=0.85

light_full
  seed=123   F1=0.810196 | IoU=0.680949 | threshold=0.7
  seed=777   F1=0.764804 | IoU=0.619176 | threshold=0.45
  seed=2024  F1=0.841149 | IoU=0.725848 | threshold=0.8

light_resized
  seed=42    F1=0.756713 | IoU=0.608639 | threshold=0.7
  seed=123   F1=0.721896 | IoU=0.564818 | threshold=0.8
  seed=777   F1=0.733585 | IoU=0.579261 | threshold=0.9

medium_full
  seed=42    F1=0.851311 | IoU=0.741116 | threshold=0.5
  seed=123   F1=0.848121 | IoU=0.736293 | threshold=0.7
  seed=777   F1=0.837745 | IoU=0.720793 | threshold=0.45

medium_resized
  seed=42    F1=0.684597 | IoU=0.520446 | threshold=0.85
  seed=123   F1=0.706518 | IoU=0.546214 | threshold=0.7
  seed=777   F1=0.661565 | IoU=0.494283 | threshold=0.65
